# EdgeGuard-Road · Semantic-First Colab

Tek giriş noktası: kurulum → veri audit/split → smoke/pilot → evaluation → ONNX → Drive sync. Gerçek veri olmadan bilimsel metrik üretmez.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPOSITORY = "https://github.com/emrealmaoglu/edgeguard-road.git"
BRANCH = "rescue/semantic-first"
PROJECT_ROOT = Path("/content/edgeguard-road")
CITYSCAPES_ROOT = Path("/content/drive/MyDrive/EdgeGuard/datasets/cityscapes")
BDD100K_ROOT = Path("/content/drive/MyDrive/EdgeGuard/datasets/bdd100k")
IDD20K_ROOT = Path("/content/drive/MyDrive/EdgeGuard/datasets/idd20k")
ACDC_ROOT = Path("/content/drive/MyDrive/EdgeGuard/datasets/acdc")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/EdgeGuard/experiments/semantic-first")
WORK_ROOT = Path("/content/edgeguard-work")
RUN_STAGE = "smoke"  # smoke, pilot, screening, final
RUN_MODELS = ["segformer_b0", "fast_scnn", "pidnet_s", "ddrnet_23_slim", "bisenetv2"]
RUN_MULTIDOMAIN_AUDIT = False  # BDD100K/IDD20K lisanslı kökleri hazırsa True
RUN_FREEZE = False  # Üretilen üç split manifesti insan tarafından incelendikten sonra True
RUN_SOURCE_VALIDATION_AUDIT = False  # Kaynak modeller dondurulduktan sonra True
RUN_FREEZE_SOURCE_VALIDATION = False  # Val auditleri ayrıca incelendikten sonra True
RUN_TRAINING = False  # Audit sonucunu incelemeden True yapmayın
RUN_HPO = False  # Screening candidate_table.json incelendikten sonra True
FINAL_MODELS = []  # İnsan tarafından dondurulmuş en fazla iki model adı
RUN_FINAL_EVALUATION = False  # Final checkpoint/config freeze sonrası True
RUN_ACDC = False
RUN_SEALED_PACKAGE = False  # Hash-bound release olmadan açılamaz
SEALED_MANIFEST = WORK_ROOT / "manifests/wilddash2.frozen.json"
SEALED_RELEASE = WORK_ROOT / "manifests/wilddash2.release.json"

In [ ]:
# ruff: noqa: E501
FINAL_PROTOCOL_CODE = r"""# Final-only calibration, withheld source validation, ACDC, and sealed packaging.
if RUN_FINAL_EVALUATION:
    if not 1 <= len(FINAL_MODELS) <= 2:
        raise RuntimeError("Freeze one or two FINAL_MODELS before final evaluation")
    source_val_manifests = {
        "bdd100k": WORK_ROOT / "manifests/official-validation/bdd100k.frozen.json",
        "idd20k": WORK_ROOT / "manifests/official-validation/idd20k.frozen.json",
    }
    for model in FINAL_MODELS:
        run_dir = RUN_ROOT / "final" / model / "ce"
        checkpoints = sorted(run_dir.glob("*.pth"))
        if not checkpoints:
            raise RuntimeError(f"Missing final checkpoint for {model}")
        checkpoint = checkpoints[-1]
        evidence = []
        for dataset, manifest in zip(("cityscapes", "bdd100k", "idd20k"), DATA_MANIFESTS, strict=True):
            target = WORK_ROOT / "calibration" / model / f"{dataset}.npz"
            if not target.exists():
                subprocess.run([str(RUNTIME_PYTHON), str(PROJECT_ROOT / "scripts/evaluate.py"), "run", "--resolved-config", str(run_dir / "resolved.py"), "--checkpoint", str(checkpoint), "--dataset", dataset, "--dataset-manifest", str(manifest), "--role", "train_calibration", "--save-calibration-evidence", str(target), "--output-dir", str(WORK_ROOT / "evaluation/calibration" / model / dataset)], check=True)
            evidence.append(target)
        temperature = WORK_ROOT / "calibration" / model / "global-temperature.json"
        if not temperature.exists():
            command = [str(RUNTIME_PYTHON), str(PROJECT_ROOT / "scripts/evaluate.py"), "calibrate-global", "--output", str(temperature)]
            for item in evidence:
                command.extend(["--evidence", str(item)])
            subprocess.run(command, check=True)
        city_output = WORK_ROOT / "evaluation/final" / model / "cityscapes"
        if not city_output.exists():
            subprocess.run([str(RUNTIME_PYTHON), str(PROJECT_ROOT / "scripts/evaluate.py"), "run", "--resolved-config", str(run_dir / "resolved.py"), "--checkpoint", str(checkpoint), "--dataset", "cityscapes", "--dataset-root", str(CITYSCAPES_ROOT), "--role", "official_val_common_eval", "--temperature-file", str(temperature), "--rare-classes-file", str(RARE), "--output-dir", str(city_output)], check=True)
        for dataset, manifest in source_val_manifests.items():
            target = WORK_ROOT / "evaluation/final" / model / dataset
            if not target.exists():
                subprocess.run([str(RUNTIME_PYTHON), str(PROJECT_ROOT / "scripts/evaluate.py"), "run", "--resolved-config", str(run_dir / "resolved.py"), "--checkpoint", str(checkpoint), "--dataset", dataset, "--dataset-manifest", str(manifest), "--role", "official_source_val", "--temperature-file", str(temperature), "--rare-classes-file", str(RARE), "--output-dir", str(target)], check=True)
        if RUN_ACDC:
            for condition in ("fog", "night", "rain", "snow"):
                target = WORK_ROOT / "evaluation/acdc" / model / condition
                if not target.exists():
                    subprocess.run([str(RUNTIME_PYTHON), str(PROJECT_ROOT / "scripts/evaluate.py"), "run", "--resolved-config", str(run_dir / "resolved.py"), "--checkpoint", str(checkpoint), "--dataset", "acdc", "--dataset-root", str(ACDC_ROOT), "--role", "domain_shift_val", "--condition", condition, "--temperature-file", str(temperature), "--rare-classes-file", str(RARE), "--output-dir", str(target)], check=True)
if RUN_SEALED_PACKAGE:
    if len(FINAL_MODELS) != 1 or not SEALED_MANIFEST.is_file() or not SEALED_RELEASE.is_file():
        raise RuntimeError("Sealed packaging requires one frozen model, manifest, and release")
    model = FINAL_MODELS[0]
    onnx_model = WORK_ROOT / "exports/final" / f"{model}.onnx"
    subprocess.run([str(RUNTIME_PYTHON), str(PROJECT_ROOT / "scripts/evaluate.py"), "package-external", "--dataset-manifest", str(SEALED_MANIFEST), "--model", str(onnx_model), "--sealed-release", str(SEALED_RELEASE), "--output-dir", str(WORK_ROOT / "external-package" / model)], check=True)
"""

In [ ]:
import subprocess
import sys

if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
PROJECT_COMMIT = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[colab]"], check=True)
print({"project_commit": PROJECT_COMMIT, "python": sys.version})

In [ ]:
# Pinned compatibility cascade; safe to rerun and records environment evidence.
install = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/train/install_semantic_stack.py"),
    "--config",
    str(PROJECT_ROOT / "configs/training/segmentation/framework_mmseg.yaml"),
    "--project-root",
    str(PROJECT_ROOT),
    "--project-commit",
    PROJECT_COMMIT,
    "--config-root",
    str(PROJECT_ROOT / "configs/training/segmentation"),
    "--runtime-current-root",
    "/content/edgeguard-runtime-current",
    "--runtime-py311-root",
    "/content/edgeguard-runtime-py311",
    "--checkout-root",
    "/content/edgeguard-checkouts",
    "--evidence-root",
    "/content/edgeguard-evidence",
    "--log-root",
    "/content/edgeguard-logs",
    "--cache-root",
    "/content/edgeguard-cache",
    "--data-root",
    str(CITYSCAPES_ROOT),
    "--execute",
]
subprocess.run(install, check=True)
RUNTIME_PYTHON = Path("/content/edgeguard-runtime-py311/bin/python")
if not RUNTIME_PYTHON.is_file():
    RUNTIME_PYTHON = Path(sys.executable)
MMSEG_ROOT = Path("/content/edgeguard-checkouts/mmseg-path-b")

In [ ]:
# Audit is the hard scientific gate. Existing output is preserved, not overwritten.
import json

AUDIT_ROOT = WORK_ROOT / "audit/cityscapes"
if not (AUDIT_ROOT / "dataset_audit/summary.json").is_file():
    subprocess.run(
        [
            str(RUNTIME_PYTHON),
            str(PROJECT_ROOT / "scripts/audit_dataset.py"),
            "--dataset-root",
            str(CITYSCAPES_ROOT),
            "--output-root",
            str(AUDIT_ROOT),
        ],
        check=True,
    )

audit = json.loads((AUDIT_ROOT / "dataset_audit/summary.json").read_text())
if not audit["audit_passed"]:
    raise RuntimeError("Dataset audit failed; scientific training is blocked")
SPLIT = AUDIT_ROOT / "dataset_audit/CSF-SPLIT-D.json"
MANIFEST_ROOT = WORK_ROOT / "manifests"
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_MULTIDOMAIN_AUDIT:
    for dataset, root in (("bdd100k", BDD100K_ROOT), ("idd20k", IDD20K_ROOT)):
        destination = WORK_ROOT / "audit" / dataset
        if not (destination / f"{dataset}_audit/summary.json").is_file():
            subprocess.run(
                [
                    str(RUNTIME_PYTHON),
                    str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                    "--dataset",
                    dataset,
                    "--dataset-root",
                    str(root),
                    "--output-root",
                    str(destination),
                ],
                check=True,
            )
if RUN_FREEZE:
    candidates = {
        "cityscapes": AUDIT_ROOT / "dataset_audit/dataset_manifest.candidate.json",
        "bdd100k": WORK_ROOT / "audit/bdd100k/bdd100k_audit/dataset_manifest.candidate.json",
        "idd20k": WORK_ROOT / "audit/idd20k/idd20k_audit/dataset_manifest.candidate.json",
    }
    for dataset, candidate in candidates.items():
        frozen = MANIFEST_ROOT / f"{dataset}.frozen.json"
        if not frozen.is_file():
            subprocess.run(
                [
                    str(RUNTIME_PYTHON),
                    str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                    "--dataset",
                    dataset,
                    "--dataset-root",
                    str(CITYSCAPES_ROOT),
                    "--output-root",
                    str(MANIFEST_ROOT),
                    "--split-manifest",
                    str(candidate),
                    "--freeze-approved",
                ],
                check=True,
            )
DATA_MANIFESTS = [
    MANIFEST_ROOT / f"{dataset}.frozen.json" for dataset in ("cityscapes", "bdd100k", "idd20k")
]
if RUN_SOURCE_VALIDATION_AUDIT:
    if not all(path.is_file() for path in DATA_MANIFESTS):
        raise RuntimeError("Official source validation audit requires frozen training manifests")
    for dataset, root in (("bdd100k", BDD100K_ROOT), ("idd20k", IDD20K_ROOT)):
        destination = WORK_ROOT / "audit" / f"{dataset}-val"
        summary = destination / f"{dataset}_val_audit/summary.json"
        if not summary.is_file():
            command = [
                str(RUNTIME_PYTHON),
                str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                "--dataset",
                dataset,
                "--source-split",
                "val",
                "--dataset-root",
                str(root),
                "--output-root",
                str(destination),
            ]
            for manifest in DATA_MANIFESTS:
                command.extend(["--source-manifest", str(manifest)])
            subprocess.run(command, check=True)
if RUN_FREEZE_SOURCE_VALIDATION:
    validation_manifest_root = MANIFEST_ROOT / "official-validation"
    for dataset in ("bdd100k", "idd20k"):
        candidate = (
            WORK_ROOT
            / "audit"
            / f"{dataset}-val/{dataset}_val_audit/dataset_manifest.candidate.json"
        )
        frozen = validation_manifest_root / f"{dataset}.frozen.json"
        if not frozen.is_file():
            subprocess.run(
                [
                    str(RUNTIME_PYTHON),
                    str(PROJECT_ROOT / "scripts/audit_dataset.py"),
                    "--dataset",
                    dataset,
                    "--output-root",
                    str(validation_manifest_root),
                    "--split-manifest",
                    str(candidate),
                    "--freeze-approved",
                ],
                check=True,
            )
STATS_ROOT = WORK_ROOT / "multidomain-statistics"
if all(path.is_file() for path in DATA_MANIFESTS) and not STATS_ROOT.exists():
    command = [
        str(RUNTIME_PYTHON),
        str(PROJECT_ROOT / "scripts/audit_dataset.py"),
        "--output-root",
        str(STATS_ROOT),
    ]
    for manifest in DATA_MANIFESTS:
        command.extend(["--data-manifest", str(manifest)])
    subprocess.run(command, check=True)
WEIGHTS = STATS_ROOT / "class_weights.json"
RARE = STATS_ROOT / "rare_classes.json"
print(json.dumps(audit, indent=2))

In [ ]:
# Five equal-protocol random-init runs over uniformly sampled source domains.
RUN_ROOT = WORK_ROOT / "runs"
if RUN_TRAINING:
    if not all(path.is_file() for path in DATA_MANIFESTS):
        raise RuntimeError("Three reviewed/frozen source manifests are required")
    for model in RUN_MODELS:
        command = [
            str(RUNTIME_PYTHON),
            str(PROJECT_ROOT / "scripts/train.py"),
            "--config",
            str(PROJECT_ROOT / "configs/rescue/semantic_first.yaml"),
            "--model",
            model,
            "--stage",
            RUN_STAGE,
            "--output-root",
            str(RUN_ROOT),
            "--mmseg-root",
            str(MMSEG_ROOT),
            "--loss",
            "ce",
        ]
        for manifest in DATA_MANIFESTS:
            command.extend(["--data-manifest", str(manifest)])
        run_dir = RUN_ROOT / RUN_STAGE / model / "ce"
        if run_dir.is_dir() and any(run_dir.iterdir()):
            command.append("--resume")
        subprocess.run(command, check=True)
else:
    print("RUN_TRAINING=False: audit tamamlandı, GPU eğitimi bilinçli olarak başlatılmadı.")

In [ ]:
# Frozen evaluation and ONNX export run only for checkpoints that exist.
for model in RUN_MODELS:
    run_dir = RUN_ROOT / RUN_STAGE / model / "ce"
    checkpoints = sorted(run_dir.glob("*.pth")) if run_dir.is_dir() else []
    if not checkpoints:
        continue
    checkpoint = checkpoints[-1]
    for dataset, manifest in zip(("cityscapes", "bdd100k", "idd20k"), DATA_MANIFESTS, strict=True):
        if not manifest.is_file():
            continue
        evaluation = WORK_ROOT / "evaluation" / RUN_STAGE / model / dataset
        if evaluation.exists():
            continue
        subprocess.run(
            [
                str(RUNTIME_PYTHON),
                str(PROJECT_ROOT / "scripts/evaluate.py"),
                "run",
                "--resolved-config",
                str(run_dir / "resolved.py"),
                "--checkpoint",
                str(checkpoint),
                "--dataset",
                dataset,
                "--dataset-manifest",
                str(manifest),
                "--role",
                "train_select",
                "--rare-classes-file",
                str(RARE),
                "--output-dir",
                str(evaluation),
            ],
            check=True,
        )
    onnx_path = WORK_ROOT / "exports" / RUN_STAGE / f"{model}.onnx"
    if not onnx_path.exists():
        subprocess.run(
            [
                str(RUNTIME_PYTHON),
                str(PROJECT_ROOT / "scripts/export_onnx.py"),
                "--resolved-config",
                str(run_dir / "resolved.py"),
                "--checkpoint",
                str(checkpoint),
                "--output",
                str(onnx_path),
                "--device",
                "cuda",
            ],
            check=True,
        )

# HPO may start only after measured selection evaluations and validated ONNX exports exist.
report_dir = WORK_ROOT / "reports" / RUN_STAGE
screening_evidence = list((WORK_ROOT / "evaluation" / RUN_STAGE).glob("**/evaluation.json"))
export_evidence = list((WORK_ROOT / "exports" / RUN_STAGE).glob("*.validation.json"))
if (
    RUN_STAGE == "screening"
    and len(screening_evidence) >= 6
    and len(export_evidence) >= 2
    and not report_dir.exists()
):
    subprocess.run(
        [
            str(RUNTIME_PYTHON),
            str(PROJECT_ROOT / "scripts/evaluate.py"),
            "summarize",
            "--evaluation-root",
            str(WORK_ROOT / "evaluation" / RUN_STAGE),
            "--export-root",
            str(WORK_ROOT / "exports" / RUN_STAGE),
            "--output-dir",
            str(report_dir),
        ],
        check=True,
    )
if RUN_HPO:
    candidate_table = WORK_ROOT / "reports/screening/candidate_table.json"
    if not candidate_table.is_file():
        raise RuntimeError("RUN_HPO requires completed screening evaluations and ONNX reports")
    command = [
        str(RUNTIME_PYTHON),
        str(PROJECT_ROOT / "scripts/train.py"),
        "--stage",
        "hpo",
        "--candidate-table",
        str(candidate_table),
        "--output-root",
        str(RUN_ROOT),
        "--mmseg-root",
        str(MMSEG_ROOT),
        "--rare-classes-file",
        str(RARE),
    ]
    for manifest in DATA_MANIFESTS:
        command.extend(["--data-manifest", str(manifest)])
    subprocess.run(command, check=True)

In [ ]:
import shutil

exec(FINAL_PROTOCOL_CODE)

# Hash-addressed Drive sync; exact-commit output is never overwritten.
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
destination = DRIVE_OUTPUT / PROJECT_COMMIT
if destination.exists():
    print("Existing exact-commit artifact directory preserved:", destination)
else:
    shutil.copytree(WORK_ROOT, destination)
    print("Synced:", destination)
print("Demo command: streamlit run", PROJECT_ROOT / "app.py")